## Instructions Générales:  Chaque binôme doit choisir un seul dataset parmi la liste ci-dessous (à télécharger depuis Kaggle). 
## L'objectif est de reproduire l'architecture en couches (Bronze -> Silver -> Gold), en appliquant les concepts de nettoyage, 
## de transformation (via l'API DataFrame) et d'analyse avancée (via Spark SQL). 
## Pour chaque jeu de données, vous devez impérativement réaliser les 5 tâches universelles décrites à la fin de ce document.

## 1. Dataset 1 : E-Commerce Behavior Data (Cosmetics) 
## a. Description : Actions des utilisateurs (achats, vues) sur une boutique de cosmétiques. 
## b. Rechercher sur Kaggle : Ecommerce Behavior Data from Cosmetics Shop 

## Tâche 1 : Initialisation et Zone Bronze (Ingestion) 
## 1. Configurez votre session Spark pour la connecter à votre instance locale de MinIO. 
. 


In [1]:
from pyspark.sql import SparkSession

In [2]:
# Configuration de la session Spark
spark = SparkSession.builder \
    .appName("TP2-HelloDataLake") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio-datalake:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Session Spark initialisée avec succès !")

Session Spark initialisée avec succès !


## 2. Téléchargez le dataset au format CSV et déposez le fichier brut dans le bucket s3a://bronze/E-com-behavior/

## Done

## 3. Chargez ce fichier dans un DataFrame Spark (avec inferSchema=True et header=True) et affichez explicitement le schéma généré.

In [3]:
# Lecture du fichier CSV
path_csv = "s3a://bronze/E-com-behavior/*.csv"

In [4]:
df_bronz = spark.read.option("header", "true").option("inferSchema", "true").csv(path_csv)

In [5]:
df_bronz.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [6]:
df_bronz.show(10, truncate=False)

+-------------------+----------------+----------+-------------------+-------------+---------+-----+---------+------------------------------------+
|event_time         |event_type      |product_id|category_id        |category_code|brand    |price|user_id  |user_session                        |
+-------------------+----------------+----------+-------------------+-------------+---------+-----+---------+------------------------------------+
|2019-12-01 00:00:00|remove_from_cart|5712790   |1487580005268456287|NULL         |f.o.x    |6.27 |576802932|51d85cb0-897f-48d2-918b-ad63965c12dc|
|2019-12-01 00:00:00|view            |5764655   |1487580005411062629|NULL         |cnd      |29.05|412120092|8adff31e-2051-4894-9758-224bfa8aec18|
|2019-12-01 00:00:02|cart            |4958      |1487580009471148064|NULL         |runail   |1.19 |494077766|c99a50e8-2fac-4c4d-89ec-41c05f114554|
|2019-12-01 00:00:05|view            |5848413   |1487580007675986893|NULL         |freedecor|0.79 |348405118|722ffea5-

In [7]:
print("Nombre total de lignes :", df_bronz.count())

Nombre total de lignes : 20692840


## Tâche 2 : Nettoyage et Zone Silver (Qualité des Données) 
## 1. Identifiez au moins deux colonnes numériques et appliquez un filtre pour éliminer les valeurs impossibles, aberrantes ou négatives (Ex : prix ≤ 0, quantité ≤ 0, durée négative). 

In [17]:
df_bronz.describe().show()

+-------+----------+------------------+--------------------+-------------------+--------+-----------------+-------------------+--------------------+
|summary|event_type|        product_id|         category_id|      category_code|   brand|            price|            user_id|        user_session|
+-------+----------+------------------+--------------------+-------------------+--------+-----------------+-------------------+--------------------+
|  count|  20692840|          20692840|            20692840|             353594|11935723|         20692840|           20692840|            20688242|
|   mean|      NULL| 5484296.739534883|1.554230182679516...|               NULL|    NULL| 8.53473548048573|5.215526635599699E8|                NULL|
| stddev|      NULL|1305715.7311086156|1.691037849175154...|               NULL|    NULL|19.38142460687464|8.744312150200728E7|                NULL|
|    min|      cart|              3752| 1487580004807082827|    accessories.bag|airnails|           -79.37

In [8]:
from pyspark.sql.functions import col, count, when

df_bronz.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_bronz.columns
]).show()

+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|  brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+
|         0|         0|         0|          0|     20339246|8757117|    0|      0|        4598|
+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+



## La colonne category_code contient plus de 20 millions de valeurs manquantes. Comme elle n'est pas utilisée dans les analyses demandées, elle a été conservée afin de ne pas réduire drastiquement le volume des données.
## La colonne user_session n'est pas utilisée dans l'analyse demandée, donc on peut ne pas la supprimer pour conserver nos lignes.

In [9]:
df_bronz.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [9]:
df_clean = df_bronz.filter(col("price") > 0)

In [11]:
print("Nombre total de lignes :", df_clean.count())

Nombre total de lignes : 20588552


In [10]:
df_clean.select("price").describe().show()

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|         20588552|
|   mean|8.578162251527665|
| stddev|19.42059611064461|
|    min|             0.05|
|    max|           327.78|
+-------+-----------------+



## 2. Traitez les valeurs manquantes : supprimez les lignes contenant des valeurs Null sur vos colonnes clés d'analyse. 

In [11]:
df_clean = df_clean.filter(col("brand").isNotNull())

In [12]:
print("Nombre total de lignes :", df_clean.count())

Nombre total de lignes : 11935702


## Nous avons supprimer presque 42% de notre dataset soit 8 757 117 lignes parce que la Tâche 4 repose sur un regroupement par marque. Les lignes sans marque ne pourront pas être intégrées à cette analyse.

In [16]:
print("Avant :", df_bronz.count())

print("Après :", df_clean.count())

Avant : 20692840
Après : 11935702


In [17]:
df_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_clean.columns
]).show()

+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+
|         0|         0|         0|          0|     11668953|    0|    0|      0|        2969|
+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+



In [19]:
df_bronz.groupBy("event_type").count().show()

+----------------+-------+
|      event_type|  count|
+----------------+-------+
|        purchase|1287007|
|            view|9657821|
|            cart|5768333|
|remove_from_cart|3979679|
+----------------+-------+



In [20]:
df_bronz.describe("price").show()

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|         20692840|
|   mean| 8.53473548048573|
| stddev|19.38142460687464|
|    min|           -79.37|
|    max|           327.78|
+-------+-----------------+



## 3. Enregistrez le DataFrame nettoyé dans le bucket s3a://silver/E-com-behavior_clean/ au format Parquet, en appliquant un partitionnement pertinent sur une colonne catégorielle (Ex : pays, année, catégorie). 

In [13]:
(
    df_clean.write
    .mode("overwrite")
    .partitionBy("event_type")
    .parquet("s3a://silver/E-com-behavior_clean/")
)

## Tâche 3 : Transformations Avancées (API DataFrame) 
## 1. Ajoutez une nouvelle colonne calculée combinant au moins deux indicateurs (Ex : chiffre_affaires = quantite * prix, ou ratio_popularite = likes / vues). 

In [14]:
df_transform = spark.read.parquet(
    "s3a://silver/E-com-behavior_clean/"
)

In [15]:
df_transform.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)
 |-- event_type: string (nullable = true)



## Première transformation : Colonne calculée
## La consigne est d'ajouter une nouvelle colonne calculée combinant au moins deux indicateurs.
## Dans notre dataset, il n'existe pas de colonne quantité. Mais nous allons créer une colonne purchase_amount qui sera définit comme suit:
## si l'événement est un achat (purchase)
## → le montant est égal au prix
## sinon
## → montant = 0
## Parce que seules les lignes correspondant à un achat génèrent du chiffre d'affaires.

In [16]:
from pyspark.sql.functions import when, col

df_transform = df_transform.withColumn(
    "purchase_amount",
    when(col("event_type") == "purchase", col("price"))
    .otherwise(0)
)

## Vérification

In [26]:
df_transform.select(
    "event_type",
    "price",
    "purchase_amount"
).show(10)

+----------+-----+---------------+
|event_type|price|purchase_amount|
+----------+-----+---------------+
|      view|10.32|            0.0|
|      view| 1.59|            0.0|
|      view| 5.24|            0.0|
|      view|40.48|            0.0|
|      view|10.32|            0.0|
|      view| 5.56|            0.0|
|      view|44.29|            0.0|
|      view|11.43|            0.0|
|      view| 5.24|            0.0|
|      view| 1.43|            0.0|
+----------+-----+---------------+
only showing top 10 rows



## 2. Manipulez les dates : Extrayez une composante temporelle (l'année, le mois ou l'heure) depuis la colonne temporelle principale du dataset vers une nouvelle colonne dédiée. 

## Le dataset couvre cinq mois.
## Nous allons extraire : année, mois, jour et heure
## Ainsi, nous aurons plus de possibilités d'analyse.

In [27]:
from pyspark.sql.functions import (
    year,
    month,
    dayofmonth,
    hour
)

In [28]:
df_transform = (
    df_transform
    .withColumn("year", year("event_time"))
    .withColumn("month", month("event_time"))
    .withColumn("day", dayofmonth("event_time"))
    .withColumn("hour", hour("event_time"))
)

## Vérification

In [29]:
df_transform.select(
    "event_time",
    "year",
    "month",
    "day",
    "hour"
).show(10, False)

+-------------------+----+-----+---+----+
|event_time         |year|month|day|hour|
+-------------------+----+-----+---+----+
|2019-12-18 12:01:30|2019|12   |18 |12  |
|2019-12-18 12:01:30|2019|12   |18 |12  |
|2019-12-18 12:01:32|2019|12   |18 |12  |
|2019-12-18 12:01:32|2019|12   |18 |12  |
|2019-12-18 12:01:32|2019|12   |18 |12  |
|2019-12-18 12:01:33|2019|12   |18 |12  |
|2019-12-18 12:01:35|2019|12   |18 |12  |
|2019-12-18 12:01:36|2019|12   |18 |12  |
|2019-12-18 12:01:40|2019|12   |18 |12  |
|2019-12-18 12:01:40|2019|12   |18 |12  |
+-------------------+----+-----+---+----+
only showing top 10 rows



## 3. Formatez proprement une colonne textuelle (Ex : mise en majuscules, extraction de sous-chaînes ou remplacement de caractères). 

## Nous allons formater brand, une colonne dont la tache 4 se portera pour standardiser/normaliser les données pour une meilleure analyse

In [30]:
from pyspark.sql.functions import upper, trim

df_transform = df_transform.withColumn(
    "brand",
    upper(trim(col("brand")))
)

## Vérification

In [35]:
df_transform.select(
    "brand"
).show(10, False)

+--------+
|brand   |
+--------+
|ONIQ    |
|MILV    |
|LIANAIL |
|JESSNAIL|
|KEUNE   |
|BLUESKY |
|JESSNAIL|
|KINETICS|
|GRATTOL |
|RUNAIL  |
+--------+
only showing top 10 rows



## Vérification du schema finale

In [32]:
df_transform.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- purchase_amount: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)



## Sauvegarde vers MinIo => silver

In [33]:
(
    df_transform.write
    .mode("overwrite")
    .partitionBy("event_type")
    .parquet("s3a://silver/E-com-behavior_transformed/")
)

## Tâche 4 (Analyse Croisée API & SQL) : Trouver le nombre total d'actions (clics, ajouts au panier, achats) et le prix moyen des produits, regroupés par marque (brand) et par type d'événement (event_type). 

## Première partie : API DataFrame

In [37]:
from pyspark.sql.functions import count, avg, round

In [38]:
df_stats = (
    df_transform
    .groupBy("brand", "event_type")
    .agg(
        count("*").alias("total_actions"),
        round(avg("price"), 2).alias("average_price")
    )
    .orderBy("brand", "event_type")
)

In [46]:
df_stats.show(10, truncate=False)

+--------+----------------+-------------+-------------+
|brand   |event_type      |total_actions|average_price|
+--------+----------------+-------------+-------------+
|AIRNAILS|cart            |27725        |3.16         |
|AIRNAILS|purchase        |7504         |2.9          |
|AIRNAILS|remove_from_cart|19641        |3.05         |
|AIRNAILS|view            |27354        |27.97        |
|ALMEA   |cart            |869          |24.16        |
|ALMEA   |purchase        |146          |25.18        |
|ALMEA   |remove_from_cart|421          |22.34        |
|ALMEA   |view            |3556         |29.91        |
|ANDREA  |cart            |54           |5.46         |
|ANDREA  |purchase        |6            |5.54         |
+--------+----------------+-------------+-------------+
only showing top 10 rows



## Deuxième partie : Spark SQL

## Nous allons créer une table temporaire car Spark SQL travaille sur des tables.
## Le DataFrame devient donc une table temporaire appelée : cosmetics

In [41]:
df_transform.createOrReplaceTempView("cosmetics")

## Requete sql

In [42]:
df_sql = spark.sql("""
SELECT
    brand,
    event_type,
    COUNT(*) AS total_actions,
    ROUND(AVG(price),2) AS average_price
FROM cosmetics
GROUP BY
    brand,
    event_type
ORDER BY
    brand,
    event_type
""")

In [47]:
df_sql.show(10, truncate=False)

+--------+----------------+-------------+-------------+
|brand   |event_type      |total_actions|average_price|
+--------+----------------+-------------+-------------+
|AIRNAILS|cart            |27725        |3.16         |
|AIRNAILS|purchase        |7504         |2.9          |
|AIRNAILS|remove_from_cart|19641        |3.05         |
|AIRNAILS|view            |27354        |27.97        |
|ALMEA   |cart            |869          |24.16        |
|ALMEA   |purchase        |146          |25.18        |
|ALMEA   |remove_from_cart|421          |22.34        |
|ALMEA   |view            |3556         |29.91        |
|ANDREA  |cart            |54           |5.46         |
|ANDREA  |purchase        |6            |5.54         |
+--------+----------------+-------------+-------------+
only showing top 10 rows



## Vérification

In [44]:
print(df_stats.count())

print(df_sql.count())

1032
1032


In [45]:
df_stats.exceptAll(df_sql).show()

+-----+----------+-------------+-------------+
|brand|event_type|total_actions|average_price|
+-----+----------+-------------+-------------+
+-----+----------+-------------+-------------+



## Tâche 5 (Zone Gold) : Stocker le résultat au format Parquet dans s3a://gold/E-com-behavior/brand_activity_stats/.

In [48]:
gold_df = df_stats

In [49]:
gold_df.show(20, truncate=False)

+--------+----------------+-------------+-------------+
|brand   |event_type      |total_actions|average_price|
+--------+----------------+-------------+-------------+
|AIRNAILS|cart            |27725        |3.16         |
|AIRNAILS|purchase        |7504         |2.9          |
|AIRNAILS|remove_from_cart|19641        |3.05         |
|AIRNAILS|view            |27354        |27.97        |
|ALMEA   |cart            |869          |24.16        |
|ALMEA   |purchase        |146          |25.18        |
|ALMEA   |remove_from_cart|421          |22.34        |
|ALMEA   |view            |3556         |29.91        |
|ANDREA  |cart            |54           |5.46         |
|ANDREA  |purchase        |6            |5.54         |
|ANDREA  |remove_from_cart|35           |5.43         |
|ANDREA  |view            |407          |5.54         |
|ARDELL  |cart            |4551         |6.41         |
|ARDELL  |purchase        |841          |6.37         |
|ARDELL  |remove_from_cart|2252         |6.44   

In [50]:
gold_df.printSchema()

root
 |-- brand: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- total_actions: long (nullable = false)
 |-- average_price: double (nullable = true)



In [51]:
(
    gold_df.write
    .mode("overwrite")
    .parquet(
        "s3a://gold/E-com-behavior/brand_activity_stats/"
    )
)

# Vérification

In [53]:
gold_check = spark.read.parquet(
    "s3a://gold/E-com-behavior/brand_activity_stats/"
)

In [54]:
gold_check.show(20, truncate=False)

+--------+----------------+-------------+-------------+
|brand   |event_type      |total_actions|average_price|
+--------+----------------+-------------+-------------+
|AIRNAILS|cart            |27725        |3.16         |
|AIRNAILS|purchase        |7504         |2.9          |
|AIRNAILS|remove_from_cart|19641        |3.05         |
|AIRNAILS|view            |27354        |27.97        |
|ALMEA   |cart            |869          |24.16        |
|ALMEA   |purchase        |146          |25.18        |
|ALMEA   |remove_from_cart|421          |22.34        |
|ALMEA   |view            |3556         |29.91        |
|ANDREA  |cart            |54           |5.46         |
|ANDREA  |purchase        |6            |5.54         |
|ANDREA  |remove_from_cart|35           |5.43         |
|ANDREA  |view            |407          |5.54         |
|ARDELL  |cart            |4551         |6.41         |
|ARDELL  |purchase        |841          |6.37         |
|ARDELL  |remove_from_cart|2252         |6.44   

In [55]:
gold_check.printSchema()

root
 |-- brand: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- total_actions: long (nullable = true)
 |-- average_price: double (nullable = true)



## Cette étape confirme que les données enregistrées sont lisibles et conformes.